# Pandas Internals


Problem
→ Represent heterogenous scientific/tabular data

Manual solution
→ nested dicts / lists

Why impractical
→ no alignment, no vectorization

Library abstraction
→ DataFrame (labeled columns)

Internal mechanism
→ Each column = NumPy array
→ Index controls alignment
→ Operations happen column-wise


## Example 1

Column-based storage

In [1]:
import pandas as pd
import numpy as np
df = pd.DataFrame({'a':[1,2,3],'b':[4.0,5.0,6.0]})
print(df.dtypes)

a      int64
b    float64
dtype: object


### Exercises (Example 1)
1. Predict output BEFORE running
2. What is the index after this operation?
3. Are there hidden NaNs introduced?
4. Is this a copy or a new allocation?
5. Complexity: what happens if dataset is 10M rows?

### Solutions (Example 1)
Each column stored separately. Mixed dtypes → no single contiguous array like NumPy.

## Example 2

Alignment demonstration (critical)

In [2]:
df1 = pd.DataFrame({'a':[1,2]}, index=[0,1])
df2 = pd.DataFrame({'a':[10,20]}, index=[1,2])
print(df1 + df2)

      a
0   NaN
1  12.0
2   NaN


### Exercises (Example 2)
1. Predict output BEFORE running
2. What is the index after this operation?
3. Are there hidden NaNs introduced?
4. Is this a copy or a new allocation?
5. Complexity: what happens if dataset is 10M rows?

### Solutions (Example 2)
Alignment by index → result index = union (0,1,2). Missing entries → NaN.

# Indexing and Selection (Where most bugs happen)


Problem
→ Select subsets reliably

Library abstraction
→ loc, iloc, boolean masks

Internal mechanism
→ boolean masks are vectorized NumPy arrays
→ fancy indexing creates copies


## Example 1

Boolean filtering

In [3]:
df = pd.DataFrame({'x':[1,3,5],'y':[2,2,2]})
res = df[df['x']>2]
print(res)

   x  y
1  3  2
2  5  2


### Exercises (Example 1)
1. Predict output BEFORE running
2. What is the index after this operation?
3. Are there hidden NaNs introduced?
4. Is this a copy or a new allocation?
5. Complexity: what happens if dataset is 10M rows?

### Solutions (Example 1)
Creates new DataFrame (copy). Index preserved.

## Example 2

loc vs iloc difference

In [4]:
df = pd.DataFrame({'x':[1,2,3]}, index=[10,20,30])
print(df.loc[10])
print(df.iloc[0])

x    1
Name: 10, dtype: int64
x    1
Name: 10, dtype: int64


### Exercises (Example 2)
1. Predict output BEFORE running
2. What is the index after this operation?
3. Are there hidden NaNs introduced?
4. Is this a copy or a new allocation?
5. Complexity: what happens if dataset is 10M rows?

### Solutions (Example 2)
loc uses labels (10). iloc uses position (0). Same here but differs if index irregular.

# GroupBy, Leakage and Aggregation


Problem
→ Summarize experimental groups

Internal mechanism
→ hash grouping
→ split-apply-combine

Critical risk
→ leakage if grouping includes target-like info


## Example 1

Basic groupby

In [5]:
df = pd.DataFrame({'group':['A','A','B'],'value':[1,2,3]})
print(df.groupby('group')['value'].mean())

group
A    1.5
B    3.0
Name: value, dtype: float64


### Exercises (Example 1)
1. Predict output BEFORE running
2. What is the index after this operation?
3. Are there hidden NaNs introduced?
4. Is this a copy or a new allocation?
5. Complexity: what happens if dataset is 10M rows?

### Solutions (Example 1)
Data split into groups, mean computed per group.

## Example 2

Leakage example

In [6]:
df = pd.DataFrame({'user':[1,1,2],'target':[10,12,20]})
print(df.groupby('user')['target'].mean())

user
1    11.0
2    20.0
Name: target, dtype: float64


### Exercises (Example 2)
1. Predict output BEFORE running
2. What is the index after this operation?
3. Are there hidden NaNs introduced?
4. Is this a copy or a new allocation?
5. Complexity: what happens if dataset is 10M rows?

### Solutions (Example 2)
If predicting 'target', this aggregation leaks future info. Must split BEFORE groupby in ML.

# Merge, Join and Data Corruption


Problem
→ Combine datasets

Internal mechanism
→ hash join

Critical risk
→ row duplication or loss


## Example 1

Basic merge

In [7]:
df1 = pd.DataFrame({'id':[1,2],'x':[10,20]})
df2 = pd.DataFrame({'id':[1,1],'y':[100,200]})
res = pd.merge(df1, df2, on='id')
print(res)

   id   x    y
0   1  10  100
1   1  10  200


### Exercises (Example 1)
1. Predict output BEFORE running
2. What is the index after this operation?
3. Are there hidden NaNs introduced?
4. Is this a copy or a new allocation?
5. Complexity: what happens if dataset is 10M rows?

### Solutions (Example 1)
Many-to-many join duplicates rows. Size increases → common real bug.

# Missing Data and Real Dataset Behavior


Problem
→ real measurements incomplete

Internal mechanism
→ NaN propagation in NumPy

Critical issue
→ silent propagation breaks models


## Example 1

NaN propagation

In [8]:
df = pd.DataFrame({'a':[1,None,3]})
print(df['a']*2)

0    2.0
1    NaN
2    6.0
Name: a, dtype: float64


### Exercises (Example 1)
1. Predict output BEFORE running
2. What is the index after this operation?
3. Are there hidden NaNs introduced?
4. Is this a copy or a new allocation?
5. Complexity: what happens if dataset is 10M rows?

### Solutions (Example 1)
NaN stays NaN. Computations do not remove missingness automatically.

# Performance and Memory


Problem
→ datasets scale to millions of rows

Internal mechanism
→ vectorized column ops fast
→ row-wise loops slow


## Example 1

Vectorization

In [9]:
df = pd.DataFrame({'a':range(1000)})
df['b'] = df['a']*2

### Exercises (Example 1)
1. Predict output BEFORE running
2. What is the index after this operation?
3. Are there hidden NaNs introduced?
4. Is this a copy or a new allocation?
5. Complexity: what happens if dataset is 10M rows?

### Solutions (Example 1)
Fast: underlying NumPy vectorized. Avoid Python loops.